# Galileo Brain v7 "Onnipotenza Locale" — Full Training

Fine-tuning del modello su dataset v7 stratificato:
- **85,966 training** examples (11 strati: replay + action + context + tutor + adversarial + multi-action + implicit + experiments + long-confused + dialect + augmented)
- **196 eval** examples (bilanciati per intent)

### Modello
- **Primario:** `unsloth/Qwen3.5-2B` con bf16 LoRA (NON QLoRA!)
- **Fallback:** `unsloth/Qwen3-4B` con QLoRA 4-bit (se Qwen3.5 non supportato)

### Target
- Intent accuracy >=95%, 0 parse errors
- 6 intents: action, circuit, code, tutor, vision, navigation
- 6 campi JSON: intent, entities, actions, needs_llm, response, llm_hint

### Deploy
- Export GGUF q4_k_m → Ollama locale su M1 8GB
- `ollama create galileo-brain -f Modelfile`

In [ ]:
# ═══ Cell 1: Installazione Unsloth + dipendenze ═══
import sys
if "google.colab" in sys.modules:
    !pip install --no-deps trl peft accelerate bitsandbytes
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps xformers triton
    !pip install transformers==4.56.2 trl==0.22.2 datasets

print("\u2705 Installazione completata!")
print("\u26a0\ufe0f  ORA RIAVVIA IL RUNTIME: Runtime \u2192 Riavvia runtime")
print("   Poi esegui dalla Cell 2 in poi")

In [ ]:
# ═══ Cell 2: Verifica supporto Qwen3.5 + Caricamento modello ═══
from unsloth import FastLanguageModel
import torch

# === STEP 1: Verifica se Qwen3.5 e' supportato ===
supported = FastLanguageModel.list_supported_models()
has_qwen35 = any("Qwen3.5" in m or "qwen3.5" in m.lower() for m in supported if isinstance(m, str))
print(f"Qwen3.5 supportato: {has_qwen35}")

# === STEP 2: Scegli modello ===
if has_qwen35:
    MODEL_NAME = "unsloth/Qwen3.5-2B"
    LOAD_4BIT = False  # bf16 LoRA (raccomandato per Qwen3.5)
    print(f"\u2705 Usando {MODEL_NAME} con bf16 LoRA (raccomandato)")
else:
    MODEL_NAME = "unsloth/Qwen3-4B"
    LOAD_4BIT = True   # QLoRA 4-bit (fallback sicuro)
    print(f"\u26a0\ufe0f  Qwen3.5 non supportato! Fallback a {MODEL_NAME} con QLoRA")

# === STEP 3: Carica modello ===
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = LOAD_4BIT,
)

print(f"\n\u2705 Modello caricato: {MODEL_NAME}")
print(f"   4-bit: {LOAD_4BIT} | Device: {model.device}")
print(f"   VRAM usata: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# ═══ Cell 3: Configurazione LoRA adapter ═══
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\u2705 LoRA configurato!")
print(f"   Parametri trainabili: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"   VRAM usata: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# ═══ Cell 4: Upload dataset v7 (86K train + 196 eval) ═══
from google.colab import files
import json

print("\ud83d\udcc1 Carica i 2 file JSONL:")
print("   1) galileo-brain-v7.jsonl          (train ~86K, 133 MB)")
print("   2) galileo-brain-v7-eval.jsonl     (eval 196)")
print("")
print("   \u26a0\ufe0f  133 MB e' grande — se l'upload e' lento:")
print("   Alternativa: monta Google Drive e copia i file")
print("   from google.colab import drive")
print("   drive.mount('/content/drive')")
print("   !cp '/content/drive/MyDrive/galileo-brain-v7*.jsonl' .")
print("")

uploaded = files.upload()

train_file = [f for f in uploaded if 'eval' not in f and f.endswith('.jsonl')][0]
eval_file  = [f for f in uploaded if 'eval' in f and f.endswith('.jsonl')][0]

for label, fname in [("TRAIN", train_file), ("EVAL", eval_file)]:
    with open(fname) as f:
        lines = f.readlines()
    first = json.loads(lines[0])
    # Verifica formato
    assistant_json = json.loads(first['messages'][2]['content'])
    fields = set(assistant_json.keys())
    expected = {'intent', 'entities', 'actions', 'needs_llm', 'response', 'llm_hint'}
    print(f"\n\u2705 {label}: {fname}")
    print(f"   Esempi: {len(lines):,}")
    print(f"   Campi JSON: {fields}")
    print(f"   Formato OK: {'\u2705' if fields == expected else '\u274c ERRORE! Mancano: ' + str(expected - fields)}")
    print(f"   Intent: {assistant_json.get('intent')}")

In [ ]:
# ═══ Cell 5: Preparazione dataset ═══
from datasets import load_dataset

train_dataset = load_dataset("json", data_files=train_file, split="train")
eval_dataset  = load_dataset("json", data_files=eval_file, split="train")

def format_chatml(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_chatml, num_proc=4)
eval_dataset  = eval_dataset.map(format_chatml, num_proc=4)

# Verifica distribuzione intent
import json as _json
intent_counts = {}
with open(train_file) as f:
    for line in f:
        msg = _json.loads(line)
        assistant = _json.loads(msg['messages'][2]['content'])
        intent = assistant.get('intent', 'unknown')
        intent_counts[intent] = intent_counts.get(intent, 0) + 1

print(f"\u2705 Dataset pronti!")
print(f"   Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")
print(f"\n\ud83d\udcca Distribuzione intent:")
for intent, count in sorted(intent_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(train_dataset)
    bar = '\u2588' * int(pct/2)
    print(f"   {intent:12} {bar} {count:,} ({pct:.1f}%)")

# Verifica che NON ci sia l'intent 'teacher'
if 'teacher' in intent_counts:
    print(f"\n\u274c ERRORE CRITICO: trovato intent 'teacher' ({intent_counts['teacher']} esempi)!")
    print("   Deve essere 'tutor', non 'teacher'. Rigenera il dataset.")
else:
    print(f"\n\u2705 Nessun intent 'teacher' trovato (corretto)")

In [ ]:
# ═══ Cell 6: Training config + avvio ═══
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Adatta batch size al modello scelto
if LOAD_4BIT:
    # QLoRA 4-bit (Qwen3-4B) — piu' VRAM-friendly
    BATCH_SIZE = 8
    GRAD_ACCUM = 2
else:
    # bf16 LoRA (Qwen3.5-2B) — usa piu' VRAM
    BATCH_SIZE = 4
    GRAD_ACCUM = 4

# 86K esempi / effective_batch = steps per epoch
effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(train_dataset) // effective_batch
EPOCHS = 2  # 86K e' gia' enorme, 2 epoche bastano
EVAL_STEPS = 500
SAVE_STEPS = 500
LR = 1e-4  # conservativo per dataset grande

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 4,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        eval_strategy = "steps",
        eval_steps = EVAL_STEPS,
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 3,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        logging_steps = 50,
        report_to = "none",
        output_dir = "galileo-brain-v7",
        seed = 3407,
    ),
)

total_steps = steps_per_epoch * EPOCHS
print(f"\u2705 Trainer v7 configurato!")
print(f"   Modello: {MODEL_NAME} ({'bf16 LoRA' if not LOAD_4BIT else 'QLoRA 4-bit'})")
print(f"   Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch} effective")
print(f"   Steps/epoch: ~{steps_per_epoch:,} | Total: ~{total_steps:,} ({EPOCHS} epochs)")
print(f"   LR: {LR} | Eval ogni {EVAL_STEPS} steps")
print(f"   bf16: {is_bfloat16_supported()} | Packing: ON")
print(f"\n   Stima tempo: ~2-4h (A100) / ~6-10h (T4)")
print(f"\n\ud83d\ude80 Esegui Cell 7 per avviare il training!")

In [ ]:
# ═══ Cell 7: TRAINING ═══
import time

print("\ud83d\ude80 Training v7 avviato...")
print(f"   Modello: {MODEL_NAME}")
print(f"   Dataset: {len(train_dataset):,} esempi (11 strati)")
print(f"   Stima: ~2-4h su A100, ~6-10h su T4\n")

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\n{'='*60}")
print(f"\u2705 TRAINING v7 COMPLETATO!")
print(f"   Tempo: {elapsed/60:.1f} min ({elapsed/3600:.1f} ore)")
print(f"   Loss finale: {stats.training_loss:.4f}")
print(f"   Steps totali: {stats.global_step}")
print(f"   VRAM picco: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")
print(f"{'='*60}")

In [ ]:
# ═══ Cell 8: Test inferenza rapido (45 casi) ═══
from unsloth import FastLanguageModel
import json

FastLanguageModel.for_inference(model)

test_cases = [
    # === AZIONI DIRETTE ===
    ("avvia la simulazione", "action"),
    ("ferma tutto", "action"),
    ("resetta il circuito", "action"),
    ("cancella tutto dalla breadboard", "action"),
    ("annulla l'ultima cosa", "action"),
    ("rifai quello che ho tolto", "action"),
    ("compila il codice", "action"),
    ("apri l'editor", "action"),
    ("chiudi il pannello codice", "action"),
    ("passa a Scratch", "action"),
    ("torna ad Arduino C++", "action"),
    ("rimetti il codice originale", "action"),
    # === NAVIGATION ===
    ("carica l'esperimento del semaforo", "navigation"),
    ("apri il manuale", "navigation"),
    ("vai avanti col montaggio", "action"),
    ("torna al passo precedente", "action"),
    # === CIRCUIT ===
    ("metti un LED rosso sulla breadboard", "circuit"),
    ("aggiungi un buzzer e un pulsante", "circuit"),
    ("togli il resistore", "circuit"),
    ("collega il LED al pin 13", "circuit"),
    ("evidenzia il condensatore", "circuit"),
    ("premi il pulsante", "circuit"),
    ("misura la tensione sul LED", "circuit"),
    ("fai la diagnosi del circuito", "action"),
    ("cosa c'e' sul circuito?", "action"),
    # === UI ===
    ("mostra la lista componenti", "action"),
    ("apri il monitor seriale", "action"),
    ("fammi vedere le scorciatoie", "action"),
    ("fammi un quiz", "action"),
    ("cerca un video sulle resistenze", "action"),
    # === CODE (needs_llm=true) ===
    ("scrivi il codice per far lampeggiare un LED", "code"),
    ("come faccio a leggere il sensore di luce?", "code"),
    # === TUTOR (needs_llm=true) ===
    ("come funziona un condensatore?", "tutor"),
    ("cos'e' la legge di Ohm?", "tutor"),
    ("spiegami la corrente elettrica", "tutor"),
    # === VISION ===
    ("guarda il mio circuito", "vision"),
    ("che errore c'e' nel mio circuito?", "vision"),
    # === ADVERSARIAL ===
    ("daje fallo anda'", "action"),
    ("fai il compile please", "action"),
    ("avvia la simulazzione", "action"),
    ("cosa succede se avvio?", "tutor"),
    ("fai quella cosa del circuito", "action"),
    ("non toccare niente!", "action"),
    # === LONG CONFUSED ===
    ("allora prof io non ho capito bene come si fa a mettere il led rosso sulla breadboard cioe' dove va esattamente?", "circuit"),
    ("scusa galileo ma il codice non funziona e non so perche' mi da errore quando compilo", "code"),
    ("ma cioe' se schiaccio play cosa succede? tipo si accende il led o no?", "tutor"),
]

# Usa lo STESSO system prompt del dataset
system_prompt = open(train_file).readline()
system_prompt = json.loads(system_prompt)['messages'][0]['content']

passed = 0
parse_errors = 0
for user_msg, expected_intent in test_cases:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    output = model.generate(
        input_ids=inputs, max_new_tokens=256,
        temperature=0.1, do_sample=True
    )
    response = tokenizer.decode(
        output[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()

    # Strip <think> tags (Qwen3.5 thinking mode)
    if "</think>" in response:
        response = response.split("</think>")[-1].strip()

    try:
        parsed = json.loads(response)
        intent = parsed.get("intent", "???")
        has_llm_hint = "llm_hint" in parsed
        ok = "\u2705" if intent == expected_intent else "\u274c"
        if intent == expected_intent:
            passed += 1
        hint_flag = "" if has_llm_hint else " [NO llm_hint!]"
        print(f"{ok} [{intent:10}] {user_msg[:55]}{hint_flag}")
    except json.JSONDecodeError:
        parse_errors += 1
        print(f"\u274c [PARSE ERR] {user_msg[:55]}")
        print(f"   Raw: {response[:120]}")

print(f"\n{'='*60}")
print(f"\ud83d\udcca Risultato: {passed}/{len(test_cases)} ({100*passed/len(test_cases):.1f}%)")
print(f"   Parse errors: {parse_errors}")
print(f"   Target: >=90% ({int(len(test_cases)*0.9)}/{len(test_cases)})")
if passed / len(test_cases) >= 0.9:
    print(f"\n\u2705 TEST SUPERATO! Procedi con Cell 9.")
else:
    print(f"\n\u274c TEST NON SUPERATO. Controlla:")
    print(f"   1. System prompt allineato?")
    print(f"   2. Training converso? (loss < 0.1?)")
    print(f"   3. Epoche sufficienti?")

In [ ]:
# ═══ Cell 9: Eval suite completa (196 esempi) ═══
import json

eval_results = {"correct": 0, "wrong": 0, "parse_error": 0, "by_intent": {}}

with open(eval_file) as f:
    eval_examples = [json.loads(line) for line in f]

print(f"\ud83e\uddea Eval su {len(eval_examples)} esempi...\n")

for i, example in enumerate(eval_examples):
    messages = example["messages"][:2]  # system + user
    expected = json.loads(example["messages"][2]["content"])
    expected_intent = expected["intent"]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    output = model.generate(
        input_ids=inputs, max_new_tokens=256,
        temperature=0.1, do_sample=True
    )
    response = tokenizer.decode(
        output[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()

    if "</think>" in response:
        response = response.split("</think>")[-1].strip()

    if expected_intent not in eval_results["by_intent"]:
        eval_results["by_intent"][expected_intent] = {"correct": 0, "total": 0}
    eval_results["by_intent"][expected_intent]["total"] += 1

    try:
        parsed = json.loads(response)
        if parsed.get("intent") == expected_intent:
            eval_results["correct"] += 1
            eval_results["by_intent"][expected_intent]["correct"] += 1
        else:
            eval_results["wrong"] += 1
            print(f"  \u274c [{i}] expected={expected_intent} got={parsed.get('intent')} | {messages[1]['content'][:60]}")
    except json.JSONDecodeError:
        eval_results["parse_error"] += 1
        print(f"  \ud83d\udca5 [{i}] PARSE ERROR | {response[:80]}")

    if (i+1) % 50 == 0:
        print(f"  ... {i+1}/{len(eval_examples)} done")

total = eval_results["correct"] + eval_results["wrong"] + eval_results["parse_error"]
acc = eval_results["correct"] / total * 100

print(f"\n{'='*60}")
print(f"\ud83d\udcca EVAL RESULTS v7")
print(f"   Intent accuracy: {eval_results['correct']}/{total} ({acc:.1f}%)")
print(f"   Wrong intent:    {eval_results['wrong']}")
print(f"   Parse errors:    {eval_results['parse_error']}")
print(f"\n\ud83d\udccb Per-intent breakdown:")
for intent, data in sorted(eval_results["by_intent"].items()):
    pct = data['correct'] / data['total'] * 100 if data['total'] > 0 else 0
    bar = '\u2588' * int(pct/5) + '\u2591' * (20 - int(pct/5))
    print(f"   {intent:12} {bar} {data['correct']:3}/{data['total']:3} ({pct:.0f}%)")
print(f"\n   Target: >=95% intent accuracy")
if acc >= 95:
    print(f"\n\u2705 EVAL SUPERATA! Procedi con l'export GGUF (Cell 10).")
else:
    print(f"\n\u26a0\ufe0f  Accuracy sotto il target. Considera:")
    print(f"   - Aumentare epoche a 3")
    print(f"   - Ridurre learning rate a 5e-5")

In [ ]:
# ═══ Cell 10: Export GGUF q4_k_m ═══
print("\ud83d\udce6 Esportazione GGUF q4_k_m...")
print("   Questo richiede ~5-10 minuti\n")

model.save_pretrained_gguf(
    "galileo-brain-v7-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

print("\n\u2705 GGUF esportato!")

import os, glob as g
gguf_dirs = g.glob("galileo-brain-v7-gguf*")
for d in gguf_dirs:
    if os.path.isdir(d):
        for f in sorted(os.listdir(d)):
            size_mb = os.path.getsize(os.path.join(d, f)) / 1024 / 1024
            if size_mb > 1:
                print(f"   {d}/{f}: {size_mb:.1f} MB")

In [ ]:
# ═══ Cell 11: Download GGUF + Modelfile ═══
import os, glob as g
from google.colab import files

# Trova il file GGUF
gguf_files = g.glob("galileo-brain-v7-gguf*/*.gguf")
modelfiles = g.glob("galileo-brain-v7-gguf*/Modelfile")

if gguf_files:
    gguf_path = gguf_files[0]
    size_gb = os.path.getsize(gguf_path) / 1024**3
    print(f"\u2705 GGUF: {gguf_path} ({size_gb:.2f} GB)")
    print(f"\n\u2b07\ufe0f  Scaricando GGUF...")
    files.download(gguf_path)

if modelfiles:
    print(f"\n\u2b07\ufe0f  Scaricando Modelfile...")
    files.download(modelfiles[0])
    with open(modelfiles[0]) as f:
        print(f"\n\ud83d\udcc4 Modelfile:")
        print(f.read()[:500])

print("\n" + "="*60)
print("\ud83c\udf89 TRAINING v7 COMPLETO!")
print("")
print("Prossimi passi sul Mac:")
print("  1. Installa Ollama: brew install ollama")
print("  2. Copia il .gguf in ~/models/galileo-brain-v7/")
print("  3. ollama create galileo-brain -f Modelfile")
print("  4. ollama run galileo-brain 'avvia la simulazione'")
print("  5. python3 scripts/test-brain-complete.py --model galileo-brain --full --report")
print("")
print("Target: >=95% sul test completo (120 eval + 80 stress)")